# 07 · Test-time augmentation, ensembling and the full ablation

Every row of the ablation is scored by the same `score_all` on the same test
split and 39 targets, so differences are attributable to the single change named
in the row. All thresholds and blend weights are chosen on **validation** only.

Requires the outputs of notebooks 04 (cascade probabilities), 05 (flat) and 06
(label-correlation).

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # or: pip install -e ..

import numpy as np, pandas as pd, torch
from fundus import config, labels
from fundus.data import (load_split, unhealthy_only, FundusDataset, make_loader,
                         train_transform, eval_transform, pos_weight)
from fundus.engine import set_seed, device, fit, predict

P = config.Paths()          # reads $FUNDUS_DATA_ROOT and $FUNDUS_RUN_ROOT
set_seed(config.SEED)
DEV = device()
print("device:", DEV, "| data:", P.data_root.resolve())

In [ ]:
from fundus.models import build_classifier, LabelCorrelationNet, load_weights
from fundus.inference import predict_tta, sweep_blend_weight
from fundus.metrics import tune_thresholds, score_all, per_class_table

TG, R = labels.TARGETS, P.run_root
va, te = load_split(P.csv("val")), load_split(P.csv("test"))
tf = eval_transform(config.IMG_SIZE)
val_ds, test_ds = (FundusDataset(d, P.images(s), tf, label_cols=TG) for d, s in ((va, "val"), (te, "test")))
val_dl, test_dl = make_loader(val_ds, 4, False), make_loader(test_ds, 4, False)

flat = load_weights(build_classifier(len(TG), config.FLAT), R / "flat/best_macro_ap.pt", DEV)
lc = load_weights(LabelCorrelationNet(len(TG), config.LABEL_CORR), R / "label_correlation/best_macro_ap.pt", DEV)

ABL = []
add = lambda r: (ABL.append(r), print(pd.Series(r).round(4).to_dict()))

## Flat model: plain, TTA, TTA + re-tuned thresholds

In [ ]:
Yv, Pv_flat, _ = predict(flat, val_dl, DEV)
Yt, Pt_flat, _ = predict(flat, test_dl, DEV)
TH_flat = tune_thresholds(Yv, Pv_flat)
add(score_all(Yt, Pt_flat, TH_flat, "Flat baseline"))

_, Pv_flat_tta = predict_tta(flat, val_dl, DEV)
_, Pt_flat_tta = predict_tta(flat, test_dl, DEV)
add(score_all(Yt, Pt_flat_tta, TH_flat, "Flat + TTA x4"))
TH_flat_tta = tune_thresholds(Yv, Pv_flat_tta)          # TTA shifts the probabilities
add(score_all(Yt, Pt_flat_tta, TH_flat_tta, "Flat + TTA x4 + retuned thresholds"))

## Cascade and cascade + flat

The cascade's gated zeros make its probabilities unsuitable for threshold tuning,
so it is scored at 0.5 and blended at equal weight.

In [ ]:
Pt_pipe = pd.read_csv(R / "pipeline_test/pipeline_test_probabilities.csv", index_col=0).loc[test_ds.names, TG].values
add(score_all(Yt, Pt_pipe, 0.5, "3-stage pipeline"))
add(score_all(Yt, 0.5 * Pt_pipe + 0.5 * Pt_flat_tta, TH_flat_tta, "Pipeline + flat ensemble"))

## Label-correlation model, with and without TTA

In [ ]:
_, Pv_lc, _ = predict(lc, val_dl, DEV)
_, Pt_lc, _ = predict(lc, test_dl, DEV)
add(score_all(Yt, Pt_lc, tune_thresholds(Yv, Pv_lc), "Label-correlation"))

_, Pv_lc_tta = predict_tta(lc, val_dl, DEV)
_, Pt_lc_tta = predict_tta(lc, test_dl, DEV)
TH_lc_tta = tune_thresholds(Yv, Pv_lc_tta)
add(score_all(Yt, Pt_lc_tta, TH_lc_tta, "Label-correlation + TTA x4"))

## Ensembles

In [ ]:
P3 = (Pt_pipe + Pt_flat_tta + Pt_lc_tta) / 3
add(score_all(Yt, P3, TH_lc_tta, "Three-way ensemble"))

W = sweep_blend_weight(Pv_lc_tta, Pv_flat_tta, Yv)      # weight on the label-correlation model
print(f"validation-selected weight: label-correlation {W:.2f} / flat {1 - W:.2f}")
Pv_ens = W * Pv_lc_tta + (1 - W) * Pv_flat_tta
Pt_ens = W * Pt_lc_tta + (1 - W) * Pt_flat_tta
TH_ens = tune_thresholds(Yv, Pv_ens)
add(score_all(Yt, Pt_ens, TH_ens, f"Flat + label-corr. ensemble + TTA (w_lc={W:.2f})"))

## Ablation table and exported predictions

In [ ]:
OUT = P.run("ablation")
abl = pd.DataFrame(ABL).set_index("config")
print(abl.round(4).to_string())
abl.to_csv(OUT / "ablation_table.csv")

tag = f"ensemble_flat_lc_w{int(round(W * 100)):03d}_tta"
pd.DataFrame((Pt_ens >= TH_ens).astype(int), index=test_ds.names, columns=TG) \
  .to_csv(OUT / f"test_pred_{tag}.csv", index_label="image")
pd.DataFrame(Pt_ens, index=test_ds.names, columns=[f"{c}_prob" for c in TG]) \
  .to_csv(OUT / f"test_prob_{tag}.csv", index_label="image")
per_class_table(Yt, Pt_ens, TH_ens, TG).to_csv(OUT / "test_per_disease_ensemble_flat_lc.csv")